In [9]:
import os
import subprocess
from pathlib import Path
import nbformat
from nbconvert import PythonExporter

# Define paths
base_dir = Path.home() / "drg-pipeline" / "data-cleaning"
debug_dir = base_dir / "debug" / "codebase"
temp_dir = debug_dir / "temp"
r_scripts_dir = base_dir / "r_scripts_v2"

# Ensure temp directory exists
temp_dir.mkdir(parents=True, exist_ok=True)

# Define output files
helper_scripts_path = debug_dir / "helper-scripts.R"
main_script_r_path = debug_dir / "main-script-partial-cleaning-grouping.R"
main_script_py_path = debug_dir / "main-script-grouping-python.py"
instructions_path = debug_dir / "instructions.txt"

# List of Jupyter notebooks to convert
notebooks = [
    "00b-drg-partial.ipynb",
    "02b-drg-grouping-py-v2.ipynb",
    "01-drg-cleaning-v2.ipynb",
    "02-drg-grouping-v2.ipynb"
]

# Convert Jupyter notebooks to scripts
for nb in notebooks:
    nb_path = base_dir / nb
    out_path = temp_dir / f"{nb_path.stem}.py"

    if nb_path.exists():
        with open(nb_path, "r", encoding="utf-8") as f:
            nb_content = nbformat.read(f, as_version=4)

        # Detect if it's an R notebook via metadata
        if nb_content.get("metadata", {}).get("kernelspec", {}).get("language", "") == "R":
            out_path = temp_dir / f"{nb_path.stem}.R"
            script_content = "\n\n".join(
                cell["source"].strip() for cell in nb_content["cells"] if cell["cell_type"] == "code"
            )
        else:
            script_content, _ = PythonExporter().from_notebook_node(nb_content)

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(script_content)

# Read & clean R scripts
def clean_lines(lines):
    return [line for line in lines if line.strip() and not line.strip().startswith("#")]

# Process helper R scripts
helper_content = []
for r_file in sorted(r_scripts_dir.glob("*.R")):
    helper_content.extend(clean_lines(r_file.read_text().splitlines()))
helper_scripts_path.write_text("\n".join(helper_content))

# Process main R script
r_script_files = ["00b-drg-partial.R", "01-drg-cleaning-v2.R", "02-drg-grouping-v2.R"]
main_r_content = []
for r_file in r_script_files:
    path = temp_dir / r_file
    if path.exists():
        main_r_content.extend(clean_lines(path.read_text().splitlines()))
main_script_r_path.write_text("\n".join(main_r_content))

# Process main Python script
py_script_path = temp_dir / "02b-drg-grouping-py-v2.py"
if py_script_path.exists():
    main_script_py_path.write_text("\n".join(clean_lines(py_script_path.read_text().splitlines())))

# Generate instructions.txt before deleting scripts
instructions_content = f"""Codebase Context

Helper Scripts:
{helper_scripts_path.read_text()}

R notebooks:
{main_script_r_path.read_text()}

Python notebook:
{main_script_py_path.read_text()}
"""
instructions_path.write_text(instructions_content)

# Delete the script files after saving instructions.txt
helper_scripts_path.unlink(missing_ok=True)
main_script_r_path.unlink(missing_ok=True)
main_script_py_path.unlink(missing_ok=True)

# Cleanup temp directory
subprocess.run(["rm", "-rf", str(temp_dir)])

print(f"Instructions saved to: {instructions_path}")


Instructions saved to: /home/resurreccion_cmc/drg-pipeline/data-cleaning/debug/codebase/instructions.txt
